# Building Image Generation Applications (OpenAI)

There's more to LLMs than text generation. You can also generate images from text descriptions. Images as a modality are useful across MedTech, architecture, tourism, game development, marketing, and more. In this lesson we work with today's **GPT Image** models on the OpenAI platform.

## Learning goals

By the end of this lesson you'll be able to:

- Explain what image generation is and where it's useful.
- Understand the `gpt-image` model family and how it differs from the legacy DALL·E models.
- Build an image generation application and edit images.

## What is image generation?

Image generation models create images from a text prompt. Modern models such as `gpt-image` learn the relationship between text and images during training, then iteratively turn random noise into an image that matches your prompt.

Two well-known families of image models are:

- **`gpt-image` (OpenAI)** — the current generation used in this lesson. It supports text-to-image generation and image editing (inpainting with a mask).
- **Midjourney** — a popular third-party model with its own service and Discord-based workflow.

> The older OpenAI image models — **DALL·E 2** and **DALL·E 3** — are legacy. DALL·E 3 is no longer available for new deployments, and features like `create_variation` only existed in DALL·E 2. Use the `gpt-image` models for new applications.

> **Important:** `gpt-image` models return the generated image as **base64** (`b64_json`), not as a URL. Your code decodes the base64 string to bytes and saves it — there's no image URL to download.


## Ensimmäisen kuvageneraattorisovelluksen rakentaminen

Mitä tarvitaan kuvageneraattorisovelluksen rakentamiseen? Tarvitset seuraavat kirjastot:

- **python-dotenv**, tätä kirjastoa suosittelemme vahvasti käyttämään pitämään salaisuutesi *.env*-tiedostossa erillään koodista.
- **openai**, tätä kirjastoa käytät kommunikoidaksesi OpenAI API:n kanssa.
- **pillow**, kuvien käsittelyyn Pythonissa.


1. Luo tiedosto nimeltä *.env*, jonka sisältö on seuraava:

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. Kerää yllä olevat kirjastot tiedostoon nimeltä *requirements.txt* näin:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Luo seuraavaksi virtuaaliympäristö ja asenna kirjastot:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Windowsille käytä seuraavia komentoja luodaksesi ja aktivoidaksesi virtuaaliympäristösi:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Lisää seuraava koodi tiedostoon nimeltä *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # import dotenv
    dotenv.load_dotenv()

    # Create OpenAI object (reads OPENAI_API_KEY from your .env)
    client = openai.OpenAI()


    try:
        # Create an image by using the image generation API
        generation_response = client.images.generate(
            model="gpt-image-2",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Enter your prompt text here
            size='1024x1024',
            n=1
        )
        # Set the directory for the stored image
        image_dir = os.path.join(os.curdir, 'images')

        # If the directory doesn't exist, create it
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Initialize the image path (note the filetype should be png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image models return the image as base64 (b64_json), not a URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Display the image in the default image viewer
        image = Image.open(image_path)
        image.show()

    # catch exceptions
    except openai.BadRequestError as err:
        print(err)

    ```

Selitetään tämä koodi:

- Ensin tuomme tarvitsemmamme kirjastot, mukaan lukien OpenAI-kirjaston, dotenv-kirjaston, base64-moduulin ja Pillow-kirjaston.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- Tämän jälkeen luomme clientin, joka lukee API-avaimen ``.env`` tiedostostasi.

    ```python
    # Create OpenAI object
    client = openai.OpenAI()
    ```

- Seuraavaksi generoimme kuvan:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-2",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Enter your prompt text here
        size='1024x1024',
        n=1
    )
    ```

    `gpt-image` -mallit palauttavat kuvan **base64**-merkkijonona `data[0].b64_json`. Dekoodamme sen tavuiksi ja kirjoitamme tiedostoon — lataus-URL:ia ei ole.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Lopuksi avaamme kuvan ja käytämme vakio-kuvien katseluohjelmaa näyttämään sen:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Lisätietoa kuvan generoinnista

Käydään läpi `images.generate` parametreja:

- **model**, on käytettävä kuvamalli (esimerkiksi `gpt-image-2`).
- **prompt**, on tekstikehote, jolla kuva generoidaan. Tässä se on "Jänis hevosella, pitää lollipoppia, sumuisella niityllä missä kasvaa narsisseja".
- **size**, on generoidun kuvan koko (esimerkiksi `1024x1024`, `1536x1024`, `1024x1536` tai `"auto"`).
- **n**, on generoitu kuvien lukumäärä. Tässä generoidaan yksi.

> Kuvamallit eivät ota `temperature` parametria — se on tekstin generoinnin ohjaus. Saat erilaisia tuloksia soittamalla rajapintaa uudelleen; saat yhtenäisemmän tuloksen tekemällä kehotteesta tarkemman.

## Kuvageneroinnin lisäominaisuudet

Olet nähnyt, miten kuva generoidaan muutamalla Python-rivillä. `gpt-image` -mallit voivat myös **muokata** olemassa olevaa kuvaa. Antamalla kuvan, valinnaisen **maskin** (joka osoittaa muutettavan alueen) ja kehotteen, voit muuttaa osaa kuvasta — esimerkiksi lisätä hatun meidän jänikseemme.

```python
response = client.images.edit(
  model="gpt-image-2",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# edits are also returned as base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

Peruskuvassa näkyy pelkkä jänis; lopullisessa kuvassa jäniksellä on hattu.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastuuvapauslauseke**:
Tämä asiakirja on käännetty käyttämällä tekoälypohjaista käännöspalvelua [Co-op Translator](https://github.com/Azure/co-op-translator). Vaikka pyrimme tarkkuuteen, otathan huomioon, että automaattiset käännökset saattavat sisältää virheitä tai epätarkkuuksia. Alkuperäinen asiakirja sen alkuperäiskielellä on virallinen lähde. Tärkeissä asioissa suositellaan ammattimaista ihmiskäännöstä. Emme ole vastuussa tämän käännöksen käytöstä aiheutuvista väärinymmärryksistä tai tulkinnoista.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
